# Εργαστήριο 8 — Γενετικοί Αλγόριθμοι (Genetic Algorithms)

**Μάθημα:** Ευφυή Συστήματα και Συστήματα Υποστήριξης Αποφάσεων  
**Πανεπιστήμιο Δυτικής Αττικής (ΠΑΔΑ)**

---

## Στόχοι Εργαστηρίου

Σε αυτό το εργαστήριο θα:

- Υλοποιήσουμε τους βασικούς τελεστές ενός **Γενετικού Αλγορίθμου (GA)** από μηδέν
- Επιλύσουμε πρόβλημα αριθμητικής βελτιστοποίησης (**f(x) = x²**, Goldberg 1989)
- Εφαρμόσουμε GA στο **Πρόβλημα Σακιδίου (0/1 Knapsack)**
- Εφαρμόσουμε GA στο **Πρόβλημα Πλανόδιου Πωλητή (TSP)** με Ordered Crossover (OX)
- Οπτικοποιήσουμε τη **σύγκλιση** του αλγορίθμου ανά γενιά
- Συγκρίνουμε την επίδραση των **υπερπαραμέτρων** (μέγεθος πληθυσμού, mutation rate)

---

## Απαιτούμενες Βιβλιοθήκες

| Βιβλιοθήκη | Χρήση |
|---|---|
| `numpy` | Αριθμητικές πράξεις, αποστάσεις |
| `matplotlib` | Οπτικοποίηση σύγκλισης και διαδρομών |
| `random` | Τυχαία αρχικοποίηση, επιλογή, μετάλλαξη |
| `pandas` | Εμφάνιση πινάκων αποτελεσμάτων |

```bash
pip install numpy matplotlib pandas
```

In [ ]:
pip install numpy matplotlib pandas

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

print('✓ Όλες οι βιβλιοθήκες φορτώθηκαν επιτυχώς.')

---

## 1. Βασικοί Τελεστές Γενετικού Αλγορίθμου

Πριν τρέξουμε οποιοδήποτε παράδειγμα, υλοποιούμε τις βασικές δομικές μονάδες:

| Συνάρτηση | Τελεστής | Ρόλος |
|---|---|---|
| `decode_binary` | — | Μετατρέπει δυαδικό χρωμόσωμα `[1,0,1,…]` → ακέραιο x |
| `roulette_selection` | Επιλογή | Επιλέγει γονέα ανάλογα με fitness (τροχός ρουλέτας) |
| `single_point_crossover` | Crossover | Ανταλλαγή γενετικού υλικού σε ένα τυχαίο σημείο |
| `bit_flip_mutation` | Mutation | Αντιστροφή bits με πιθανότητα `p_mut` |

Αυτές οι συναρτήσεις θα χρησιμοποιηθούν αυτούσιες στα Παραδείγματα 1 και 2.

In [ ]:
# ── Βασικοί τελεστές GA (δυαδική κωδικοποίηση) ────────────────────────────

def decode_binary(chromosome):
    """Μετατρέπει δυαδική λίστα [1,0,1,…] σε ακέραιο."""
    return int(''.join(map(str, chromosome)), 2)


def roulette_selection(population, fitnesses):
    """Επιλέγει ένα άτομο με πιθανότητα ανάλογη του fitness του."""
    total = sum(fitnesses)
    if total == 0:
        return random.choice(population)
    probs = [f / total for f in fitnesses]
    return random.choices(population, weights=probs, k=1)[0]


def single_point_crossover(p1, p2):
    """Single-point crossover — επιστρέφει δύο παιδιά."""
    point = random.randint(1, len(p1) - 1)
    c1 = p1[:point] + p2[point:]
    c2 = p2[:point] + p1[point:]
    return c1, c2


def bit_flip_mutation(chromosome, p_mut=0.05):
    """Αντιστρέφει κάθε bit με πιθανότητα p_mut."""
    return [1 - b if random.random() < p_mut else b for b in chromosome]


# ── Γρήγορος έλεγχος ──────────────────────────────────────────────────────
chrom = [0, 1, 1, 0, 1]  # 0*16 + 1*8 + 1*4 + 0*2 + 1*1 = 13
x = decode_binary(chrom)
print(f'Χρωμόσωμα       : {chrom}')
print(f'Αποκωδικοποίηση : x = {x}   (αναμένεται 13)')
print(f'Fitness f(x)=x² : {x**2}  (αναμένεται 169)')

print()
p1 = [1, 0, 0, 1, 1]
p2 = [0, 1, 1, 0, 0]
random.seed(1)
c1, c2 = single_point_crossover(p1, p2)
print(f'Crossover  P1={p1}  P2={p2}')
print(f'  → Παιδί 1: {c1}')
print(f'  → Παιδί 2: {c2}')

---

## 2. Παράδειγμα 1 — Μεγιστοποίηση f(x) = x²

**Πρόβλημα:** Εύρεση ακέραιου $x \in \{0, \ldots, 31\}$ που μεγιστοποιεί $f(x) = x^2$.  
Η βέλτιστη απάντηση είναι **x = 31**, f(31) = 961.

**Κωδικοποίηση:** Κάθε άτομο = **5-bit δυαδικό χρωμόσωμα** (Goldberg, 1989).  
Παράδειγμα: `01101` → $0{\times}16 + 1{\times}8 + 1{\times}4 + 0{\times}2 + 1{\times}1 = 13$, $f(13)=169$.

| Χρωμόσωμα | Αποκωδ. x | f(x) = x² | P(επιλογής) ≈ |
|:---:|:---:|:---:|:---:|
| `01101` | 13 | 169 | 14,4% |
| `11000` | 24 | 576 | 49,2% |
| `01000` | 8 | 64 | 5,5% |
| `10011` | 19 | 361 | 30,9% |

> **Πίεση Επιλογής:** Το Άτομο 2 έχει 3,4× μεγαλύτερο fitness από το Άτομο 1 → εμφανίζεται 3,4× πιο συχνά στην επόμενη γενιά.

In [ ]:
# ── GA για f(x) = x² ──────────────────────────────────────────────────────

BITS       = 5    # μήκος χρωμοσώματος
POP_SIZE   = 10   # μέγεθος πληθυσμού
GENERATIONS = 25  # αριθμός γενιών
P_CROSS    = 0.8  # πιθανότητα crossover
P_MUT      = 0.05 # πιθανότητα μετάλλαξης ανά bit


def fitness_fx2(chromosome):
    return decode_binary(chromosome) ** 2


def run_ga_binary(pop_size=POP_SIZE, generations=GENERATIONS,
                  p_cross=P_CROSS, p_mut=P_MUT):
    # Αρχικός τυχαίος πληθυσμός
    population = [[random.randint(0, 1) for _ in range(BITS)]
                  for _ in range(pop_size)]

    best_per_gen = []
    avg_per_gen  = []

    for _ in range(generations):
        fitnesses = [fitness_fx2(ind) for ind in population]
        best_per_gen.append(max(fitnesses))
        avg_per_gen.append(sum(fitnesses) / len(fitnesses))

        # Ελιτισμός: το καλύτερο περνάει αυτόματα
        best_idx = fitnesses.index(max(fitnesses))
        new_pop = [population[best_idx][:]]

        while len(new_pop) < pop_size:
            p1 = roulette_selection(population, fitnesses)
            p2 = roulette_selection(population, fitnesses)
            if random.random() < p_cross:
                c1, c2 = single_point_crossover(p1, p2)
            else:
                c1, c2 = p1[:], p2[:]
            new_pop.append(bit_flip_mutation(c1, p_mut))
            new_pop.append(bit_flip_mutation(c2, p_mut))

        population = new_pop[:pop_size]

    fitnesses = [fitness_fx2(ind) for ind in population]
    best = population[fitnesses.index(max(fitnesses))]
    return best, best_per_gen, avg_per_gen


random.seed(42)
best_chrom, best_hist, avg_hist = run_ga_binary()
x_best = decode_binary(best_chrom)
print(f'Βέλτιστο χρωμόσωμα : {best_chrom}')
print(f'Αποκωδικοποίηση    : x = {x_best}')
print(f'Fitness            : f({x_best}) = {x_best**2}  (βέλτιστο: 961)')

In [ ]:
# ── Αρχικός Πληθυσμός (Γενιά 0) — Goldberg 1989 ──────────────────────────

goldberg_pop = [
    [0, 1, 1, 0, 1],  # 13
    [1, 1, 0, 0, 0],  # 24
    [0, 1, 0, 0, 0],  # 8
    [1, 0, 0, 1, 1],  # 19
]

rows = []
total_fit = sum(decode_binary(c)**2 for c in goldberg_pop)
for c in goldberg_pop:
    x = decode_binary(c)
    f = x ** 2
    rows.append({
        'Χρωμόσωμα': ''.join(map(str, c)),
        'x': x,
        'f(x)=x²': f,
        'P(επιλογής)': f'{f/total_fit*100:.1f}%',
        'Αναμ. αντίγρ.': f'{f/total_fit*4:.2f}',
    })

df = pd.DataFrame(rows)
print('Αρχικός Πληθυσμός (Goldberg 1989):')
print(df.to_string(index=False))
print(f'\nΣύνολο fitness: {total_fit}')

In [ ]:
# ── Γράφημα Σύγκλισης f(x)=x² ────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(best_hist, 'b-o', markersize=5, linewidth=2, label='Καλύτερο fitness')
ax.plot(avg_hist,  'r--s', markersize=4, linewidth=1.5, label='Μέσο fitness')
ax.axhline(961, color='gray', linestyle=':', linewidth=1.2, label='Βέλτιστο (961)')
ax.set_xlabel('Γενιά')
ax.set_ylabel('Fitness  f(x) = x²')
ax.set_title('Σύγκλιση GA — Μεγιστοποίηση f(x) = x²')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 3. Παράδειγμα 2 — Πρόβλημα Σακιδίου (0/1 Knapsack)

**Πρόβλημα:** Επιλέξτε αντικείμενα που χωρούν σε σακίδιο **χωρητικότητας 10 kg**,  
μεγιστοποιώντας τη **συνολική αξία (€)**.

**Κωδικοποίηση:** Κάθε άτομο = **δυαδικό χρωμόσωμα** μήκους 5 (ένα bit ανά αντικείμενο).  
`1` = επιλέγεται · `0` = αποκλείεται.  
Χρωμόσωμα `[1, 0, 1, 1, 0]` → Laptop + Κάμερα + Tablet = 10 kg, 19 €.

| Αντικείμενο | Βάρος (kg) | Αξία (€) | Αξία/kg |
|:---:|:---:|:---:|:---:|
| Laptop | 3 | 9 | 3,0 |
| Βιβλίο | 4 | 5 | 1,25 |
| Κάμερα | 2 | 7 | 3,5 |
| Tablet | 5 | 3 | 0,6 |
| Ρολόι | 1 | 4 | 4,0 |

**Fitness:** αξία αντικειμένων αν βάρος ≤ 10 kg, αλλιώς **0** (ποινή υπέρβασης).

In [ ]:
# ── Ορισμός και επίλυση Knapsack ──────────────────────────────────────────

ITEMS = [
    {'name': 'Laptop',  'weight': 3, 'value': 9},
    {'name': 'Βιβλίο',  'weight': 4, 'value': 5},
    {'name': 'Κάμερα',  'weight': 2, 'value': 7},
    {'name': 'Tablet',  'weight': 5, 'value': 3},
    {'name': 'Ρολόι',   'weight': 1, 'value': 4},
]
MAX_WEIGHT = 10
N_ITEMS = len(ITEMS)


def fitness_knapsack(chromosome):
    total_w = sum(ITEMS[i]['weight'] for i in range(N_ITEMS) if chromosome[i] == 1)
    total_v = sum(ITEMS[i]['value']  for i in range(N_ITEMS) if chromosome[i] == 1)
    return total_v if total_w <= MAX_WEIGHT else 0


def run_ga_knapsack(pop_size=30, generations=50, p_cross=0.8, p_mut=0.1):
    population = [[random.randint(0, 1) for _ in range(N_ITEMS)]
                  for _ in range(pop_size)]
    best_per_gen = []

    for _ in range(generations):
        fitnesses = [fitness_knapsack(ind) for ind in population]
        best_per_gen.append(max(fitnesses))

        best_idx = fitnesses.index(max(fitnesses))
        new_pop = [population[best_idx][:]]

        while len(new_pop) < pop_size:
            p1 = roulette_selection(population, fitnesses)
            p2 = roulette_selection(population, fitnesses)
            if random.random() < p_cross:
                c1, c2 = single_point_crossover(p1, p2)
            else:
                c1, c2 = p1[:], p2[:]
            new_pop.append(bit_flip_mutation(c1, p_mut))
            new_pop.append(bit_flip_mutation(c2, p_mut))

        population = new_pop[:pop_size]

    fitnesses = [fitness_knapsack(ind) for ind in population]
    best = population[fitnesses.index(max(fitnesses))]
    return best, best_per_gen


random.seed(42)
best_ks, ks_hist = run_ga_knapsack()

print(f'Βέλτιστο χρωμόσωμα: {best_ks}')
print()
total_w, total_v = 0, 0
for i, item in enumerate(ITEMS):
    if best_ks[i] == 1:
        total_w += item['weight']
        total_v += item['value']
        mark = '✓'
    else:
        mark = '✗'
    print(f"  {mark}  {item['name']:<8}  {item['weight']} kg  {item['value']}€")

print(f'\nΣυνολικό βάρος : {total_w} / {MAX_WEIGHT} kg')
print(f'Συνολική αξία  : {total_v} €')

In [ ]:
# ── Γράφημα Σύγκλισης Knapsack ────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ks_hist, 'g-o', markersize=5, linewidth=2, label='Καλύτερη αξία (€)')
ax.set_xlabel('Γενιά')
ax.set_ylabel('Fitness (αξία σε €)')
ax.set_title('Σύγκλιση GA — Knapsack Problem')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 4. Παράδειγμα 3 — Πρόβλημα Πλανόδιου Πωλητή (TSP)

**Πρόβλημα:** Επισκεφθείτε 5 πόλεις (A–E) **ακριβώς μία φορά** και επιστρέψτε στην αφετηρία,  
ελαχιστοποιώντας τη **συνολική απόσταση**.

**Κωδικοποίηση:** Κάθε άτομο = **permutation** χρωμόσωμα — διάταξη των πόλεων.  
Παράδειγμα: `[A, C, B, E, D]` → A→C→B→E→D→A.

**Fitness:** $f = \frac{1}{\text{συνολική απόσταση}}$ — μικρότερη απόσταση = μεγαλύτερο fitness.

**Γιατί δεν κάνει single-point crossover;**  
Το `[A,B,C,D,E]` × `[C,A,E,B,D]` με σημείο τομής μετά τη θέση 2 θα έδινε `[A,B,E,B,D]` — η πόλη B εμφανίζεται δύο φορές! Γι' αυτό χρησιμοποιούμε **Ordered Crossover (OX)**.

### Πώς λειτουργεί το OX;

1. Επιλέγουμε τυχαίο τμήμα από τον P1 (π.χ. θέσεις 1–2)
2. Αντιγράφουμε το τμήμα στο παιδί
3. Συμπληρώνουμε τις υπόλοιπες θέσεις με τη σειρά που εμφανίζονται στον P2, **παραλείποντας** τις πόλεις που υπάρχουν ήδη

| | Θέση 0 | Θέση 1 | Θέση 2 | Θέση 3 | Θέση 4 |
|:---:|:---:|:---:|:---:|:---:|:---:|
| P1 | A | **B** | **C** | D | E |
| P2 | C | A | E | B | D |
| Παιδί | A | **B** | **C** | E | D |

In [ ]:
# ── Ορισμός πόλεων ────────────────────────────────────────────────────────

CITIES = {
    'A': (1.0, 1.0),
    'B': (4.0, 3.0),
    'C': (7.0, 1.0),
    'D': (6.0, 5.0),
    'E': (2.0, 5.0),
}
CITY_NAMES = list(CITIES.keys())


def euclidean(a, b):
    return np.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2)


def total_distance(route):
    n = len(route)
    return sum(euclidean(CITIES[route[i]], CITIES[route[(i+1) % n]])
               for i in range(n))


def fitness_tsp(route):
    return 1.0 / total_distance(route)


# Πίνακας αποστάσεων
print('Πίνακας Αποστάσεων (Euclidean):')
header = '       ' + ''.join(f'{c:>7}' for c in CITY_NAMES)
print(header)
for c1 in CITY_NAMES:
    row = f'{c1:<7}' + ''.join(f'{euclidean(CITIES[c1], CITIES[c2]):>7.2f}'
                               for c2 in CITY_NAMES)
    print(row)

In [ ]:
# ── OX Crossover & Swap Mutation ─────────────────────────────────────────

def ox_crossover(p1, p2):
    """Ordered Crossover — διατηρεί έγκυρη permutation."""
    n = len(p1)
    start, end = sorted(random.sample(range(n), 2))
    child = [None] * n
    child[start:end+1] = p1[start:end+1]           # αντιγραφή από P1
    remaining = [x for x in p2 if x not in child]  # από P2, χωρίς διπλότυπα
    idx = 0
    for i in range(n):
        if child[i] is None:
            child[i] = remaining[idx]
            idx += 1
    return child


def swap_mutation(route, p_mut=0.2):
    """Ανταλλαγή δύο τυχαίων πόλεων με πιθανότητα p_mut."""
    route = route[:]
    if random.random() < p_mut:
        i, j = random.sample(range(len(route)), 2)
        route[i], route[j] = route[j], route[i]
    return route


# ── Επίδειξη OX ──────────────────────────────────────────────────────────
p1 = ['A', 'B', 'C', 'D', 'E']
p2 = ['C', 'A', 'E', 'B', 'D']
random.seed(7)  # τμήμα 1–2 τυχαία
child = ox_crossover(p1, p2)
print('OX Crossover:')
print(f'  P1    : {p1}')
print(f'  P2    : {p2}')
print(f'  Παιδί : {child}')
print(f'  Απόσταση παιδιού: {total_distance(child):.3f}')

print()
r = ['A', 'C', 'B', 'D', 'E']
random.seed(3)
r_mut = swap_mutation(r, p_mut=1.0)  # εξαναγκάζουμε μετάλλαξη
print('Swap Mutation:')
print(f'  Πριν : {r}   ({total_distance(r):.3f})')
print(f'  Μετά : {r_mut}   ({total_distance(r_mut):.3f}')

In [ ]:
# ── Εκτέλεση GA για TSP ───────────────────────────────────────────────────

def run_ga_tsp(pop_size=80, generations=300, p_cross=0.9, p_mut=0.2):
    population = [random.sample(CITY_NAMES, len(CITY_NAMES))
                  for _ in range(pop_size)]
    best_dist_per_gen = []

    for _ in range(generations):
        fitnesses = [fitness_tsp(ind) for ind in population]
        dists     = [total_distance(ind) for ind in population]
        best_dist_per_gen.append(min(dists))

        best_idx = dists.index(min(dists))
        new_pop = [population[best_idx][:]]

        while len(new_pop) < pop_size:
            p1 = roulette_selection(population, fitnesses)
            p2 = roulette_selection(population, fitnesses)
            child = ox_crossover(p1, p2) if random.random() < p_cross else p1[:]
            new_pop.append(swap_mutation(child, p_mut))

        population = new_pop[:pop_size]

    dists = [total_distance(ind) for ind in population]
    best = population[dists.index(min(dists))]
    return best, best_dist_per_gen


random.seed(42)
best_route, tsp_hist = run_ga_tsp()
print('Βέλτιστη Διαδρομή :', ' → '.join(best_route), '→', best_route[0])
print(f'Συνολική Απόσταση : {total_distance(best_route):.3f}')

# Σύγκριση με brute-force (5! = 120 διαδρομές)
from itertools import permutations
all_routes = list(permutations(CITY_NAMES))
opt_dist = min(total_distance(list(r)) for r in all_routes)
print(f'Βέλτιστο (brute-force) : {opt_dist:.3f}')
gap = (total_distance(best_route) - opt_dist) / opt_dist * 100
print(f'Απόκλιση GA από βέλτιστο: {gap:.2f}%')

In [ ]:
# ── Οπτικοποίηση TSP: Σύγκλιση + Βέλτιστη Διαδρομή ──────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Καμπύλη σύγκλισης
ax1.plot(tsp_hist, 'b-', linewidth=1.5)
ax1.axhline(opt_dist, color='red', linestyle='--', linewidth=1.2, label=f'Βέλτιστο ({opt_dist:.2f})')
ax1.set_xlabel('Γενιά')
ax1.set_ylabel('Απόσταση')
ax1.set_title('Σύγκλιση GA — TSP')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Χάρτης βέλτιστης διαδρομής
route_loop = best_route + [best_route[0]]
xs = [CITIES[c][0] for c in route_loop]
ys = [CITIES[c][1] for c in route_loop]

ax2.plot(xs, ys, 'b-o', markersize=14, linewidth=2, zorder=2)
for city, (x, y) in CITIES.items():
    ax2.annotate(city, (x, y), ha='center', va='center',
                 fontsize=12, fontweight='bold', color='white', zorder=3)

# Αποστάσεις ακμών
for i in range(len(best_route)):
    c1, c2 = route_loop[i], route_loop[i+1]
    mx = (CITIES[c1][0] + CITIES[c2][0]) / 2
    my = (CITIES[c1][1] + CITIES[c2][1]) / 2
    d  = euclidean(CITIES[c1], CITIES[c2])
    ax2.annotate(f'{d:.2f}', (mx, my), fontsize=8.5, color='darkred',
                 ha='center', bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.7))

ax2.set_title(f'Βέλτιστη Διαδρομή TSP  (d = {total_distance(best_route):.2f})')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 5. Επίδραση Υπερπαραμέτρων

Ο GA έχει αρκετές **υπερπαραμέτρους** που επηρεάζουν σημαντικά την απόδοσή του:

| Παράμετρος | Πολύ μικρή τιμή | Πολύ μεγάλη τιμή |
|---|---|---|
| `pop_size` | Γρήγορη σύγκλιση, κίνδυνος τοπικού βελτίστου | Αργός, περισσότερη εξερεύνηση |
| `p_mut` | Χωρίς εξερεύνηση → παγίδευση | Τυχαία αναζήτηση → αργή σύγκλιση |
| `p_cross` | Χωρίς ανταλλαγή γνώσης | Υπερβολική ανακάτεμα |
| `generations` | Πρόωρη διακοπή | Σπατάλη χρόνου μετά τη σύγκλιση |

Παρακάτω συγκρίνουμε τρία διαφορετικά **μεγέθη πληθυσμού** και τρία **ποσοστά μετάλλαξης**.

In [ ]:
# ── Σύγκριση υπερπαραμέτρων ───────────────────────────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Α) Μέγεθος πληθυσμού
for pop_size, color, lbl in [
    (4,  'red',   'pop=4  (πολύ μικρός)'),
    (10, 'blue',  'pop=10 (βασικός)'),
    (50, 'green', 'pop=50 (μεγάλος)'),
]:
    random.seed(42)
    _, bh, _ = run_ga_binary(pop_size=pop_size, generations=30)
    ax1.plot(bh, color=color, linewidth=1.8, label=lbl)

ax1.axhline(961, color='gray', linestyle=':', linewidth=1, label='Βέλτιστο (961)')
ax1.set_xlabel('Γενιά')
ax1.set_ylabel('Καλύτερο Fitness')
ax1.set_title('Επίδραση Μεγέθους Πληθυσμού')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Β) Ποσοστό μετάλλαξης
for p_mut, color, lbl in [
    (0.0,  'red',    'p_mut=0.0  (χωρίς μετάλλαξη)'),
    (0.05, 'blue',   'p_mut=0.05 (τυπικό)'),
    (0.5,  'orange', 'p_mut=0.5  (υψηλό)'),
]:
    random.seed(42)
    _, bh, _ = run_ga_binary(pop_size=10, generations=30, p_mut=p_mut)
    ax2.plot(bh, color=color, linewidth=1.8, label=lbl)

ax2.axhline(961, color='gray', linestyle=':', linewidth=1, label='Βέλτιστο (961)')
ax2.set_xlabel('Γενιά')
ax2.set_ylabel('Καλύτερο Fitness')
ax2.set_title('Επίδραση Ποσοστού Μετάλλαξης')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 6. Ασκήσεις

**Άσκηση 1 — Επίδραση mutation rate:**  
Τρέξτε τον `run_ga_binary` με `p_mut=0.0` (χωρίς μετάλλαξη) και `p_mut=0.5` (πολύ υψηλό).  
Σχεδιάστε τις καμπύλες σύγκλισης και εξηγήστε: γιατί το `p_mut=0.0` κολλάει σε τοπικό βέλτιστο;

**Άσκηση 2 — Tournament Selection:**  
Υλοποιήστε **tournament selection** ως εναλλακτική της ρουλέτας. Επιλέγει k τυχαία άτομα και επιστρέφει το καλύτερο.

```python
def tournament_selection(population, fitnesses, k=3):
    # Διαλέξτε k τυχαίες θέσεις
    # Επιστρέψτε το άτομο με το μεγαλύτερο fitness
    ...
```

Αντικαταστήστε τη `roulette_selection` με `tournament_selection` στο `run_ga_binary`.  
Συγκρίνετε ταχύτητα σύγκλισης με k=2 και k=5.

**Άσκηση 3 — Knapsack με νέα αντικείμενα:**  
Προσθέστε 3 νέα αντικείμενα στο `ITEMS`, αυξήστε `MAX_WEIGHT = 15 kg`.  
Συμπεριλάβετε ένα «παγιδευτικό» αντικείμενο (μεγάλο βάρος, χαμηλή αξία/kg).  
Ελέγξτε αν ο GA το αποκλείει από τη βέλτιστη λύση.

**Άσκηση 4 — TSP με περισσότερες πόλεις:**  
Προσθέστε 3 επιπλέον πόλεις (F, G, H) στο `CITIES`.  
Αυξήστε `pop_size=150` και `generations=500`.  
Συγκρίνετε το αποτέλεσμα του GA με τον **Nearest Neighbor** (greedy) αλγόριθμο:

```python
def nearest_neighbor(start, cities):
    unvisited = list(cities.keys())
    route = [start]
    unvisited.remove(start)
    while unvisited:
        last = route[-1]
        nearest = min(unvisited, key=lambda c: euclidean(cities[last], cities[c]))
        route.append(nearest)
        unvisited.remove(nearest)
    return route
```

**Άσκηση 5 (προαιρετική) — 2-opt τοπική αναζήτηση:**  
Μετά από κάθε γενιά του TSP GA, εφαρμόστε **2-opt** στο καλύτερο άτομο:  
επιλέξτε τυχαία 2 ακμές και ελέγξτε αν η αντιστροφή του μεταξύ τους τμήματος βελτιώνει τη διαδρομή.  
Αυτός ο συνδυασμός (GA + τοπική αναζήτηση) ονομάζεται **Memetic Algorithm**.